# Drugs.com Comment EDA and Coverage Check

目标：
- 对 `drugsComTrain_raw_cleaned.csv` + `drugsComTest_raw_cleaned.csv` 做 EDA
- 抽样检查 comment / review 文本，判断后续可以做哪些 feature
- 检查评论数据里的药名，能否覆盖 hybrid 推荐管线用到的药表
- 为后续是否建立 comment feature table / DB 提供依据

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1]

TRAIN_PATH = REPO_ROOT / 'app/dataset_module/kuc-hackathon-winter-2018/drugsComTrain_raw_cleaned.csv'
TEST_PATH = REPO_ROOT / 'app/dataset_module/kuc-hackathon-winter-2018/drugsComTest_raw_cleaned.csv'
DRUG_TABLE_PATH = REPO_ROOT / 'match_data_preprocessing/data/enhanced_drug_table_v1.csv'

print('TRAIN_PATH exists =', TRAIN_PATH.exists())
print('TEST_PATH exists =', TEST_PATH.exists())
print('DRUG_TABLE_PATH exists =', DRUG_TABLE_PATH.exists())

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
reviews_df = pd.concat([train_df, test_df], ignore_index=True)
drug_df = pd.read_csv(DRUG_TABLE_PATH)

reviews_df.columns = [c.strip() for c in reviews_df.columns]
drug_df.columns = [c.strip() for c in drug_df.columns]

reviews_df['drug_key'] = reviews_df['drugName'].astype(str).str.strip().str.lower()
drug_df['drug_key'] = drug_df['drug_name'].astype(str).str.strip().str.lower()
reviews_df['review_len'] = reviews_df['review'].astype(str).str.len()
reviews_df['condition_clean'] = reviews_df['condition'].fillna('')

print('reviews_df shape =', reviews_df.shape)
print('drug_df shape =', drug_df.shape)
print('review columns =', reviews_df.columns.tolist())

## 1. Basic Overview

In [ ]:
overview = pd.DataFrame({
    'metric': [
        'review_rows',
        'unique_review_drugs',
        'unique_conditions',
        'missing_condition_rows',
        'avg_review_length',
        'median_review_length',
        'avg_rating',
        'median_usefulCount',
    ],
    'value': [
        len(reviews_df),
        reviews_df['drug_key'].nunique(),
        reviews_df['condition_clean'].nunique(),
        int(reviews_df['condition'].isna().sum()),
        round(reviews_df['review_len'].mean(), 2),
        int(reviews_df['review_len'].median()),
        round(reviews_df['rating'].mean(), 3),
        int(reviews_df['usefulCount'].median()),
    ]
})
overview

In [ ]:
reviews_df[['rating', 'usefulCount', 'review_len']].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

reviews_df['rating'].plot(kind='hist', bins=10, ax=axes[0], title='Rating Distribution')
axes[0].set_xlabel('rating')

reviews_df['usefulCount'].clip(upper=200).plot(kind='hist', bins=30, ax=axes[1], title='UsefulCount Distribution (clipped at 200)')
axes[1].set_xlabel('usefulCount')

reviews_df['review_len'].clip(upper=2000).plot(kind='hist', bins=40, ax=axes[2], title='Review Length Distribution (clipped at 2000)')
axes[2].set_xlabel('review length')

plt.tight_layout()
plt.show()

In [ ]:
print('Top conditions')
display(reviews_df['condition'].value_counts(dropna=False).head(20).to_frame('review_count'))

print('Top drugs by review count')
display(reviews_df['drugName'].value_counts().head(20).to_frame('review_count'))

## 2. Comment / Review Sampling

这里先做人工抽样，观察文本里常见的信息类型：
- 疗效 / 是否起作用
- 副作用 / 不良反应
- price / worth / cost
- overall satisfaction / recommendation
- 适用人群、起效时间、持续时间、和其他药物比较

In [ ]:
def show_samples(df, n=5, random_state=42, cols=None, max_chars=700):
    cols = cols or ['drugName', 'condition', 'rating', 'usefulCount', 'review']
    sample = df.sample(n=min(n, len(df)), random_state=random_state)[cols].copy()
    sample['review'] = sample['review'].astype(str).str.slice(0, max_chars)
    return sample

show_samples(reviews_df, n=8, random_state=42)

In [ ]:
mixed_signal_mask = reviews_df['review'].astype(str).str.contains(r'\b(but|however|although|though)\b', case=False, na=False)
show_samples(reviews_df[mixed_signal_mask], n=8, random_state=7)

In [ ]:
long_reviews = reviews_df.sort_values('review_len', ascending=False)[['drugName', 'condition', 'rating', 'usefulCount', 'review_len', 'review']].head(10).copy()
long_reviews['review'] = long_reviews['review'].astype(str).str.slice(0, 1000)
long_reviews

## 3. Quick Feature Discovery

这一步不是正式 NLP，只是用关键词快速判断 review 文本里大概能抽出哪些 feature。

In [ ]:
FEATURE_KEYWORDS = {
    'efficacy': ['worked', 'works', 'effective', 'helped', 'better', 'improved', 'relief'],
    'side_effects': ['side effect', 'nausea', 'headache', 'vomiting', 'dizzy', 'bleeding', 'rash', 'painful'],
    'cost': ['expensive', 'cheap', 'cost', 'price', 'worth', 'money', 'afford'],
    'overall': ['recommend', 'love', 'hate', 'best', 'worst', 'amazing', 'terrible', 'good', 'bad'],
    'time_to_effect': ['day 1', 'days', 'weeks', 'months', 'immediately', 'right away'],
    'comparison': ['better than', 'worse than', 'compared to', 'unlike', 'instead of'],
}

for feature, keywords in FEATURE_KEYWORDS.items():
    pattern = '|'.join(re.escape(k) for k in keywords)
    hit_rate = reviews_df['review'].astype(str).str.contains(pattern, case=False, na=False).mean()
    print(f'{feature:15s} hit_rate = {hit_rate:.3f}')

In [ ]:
feature_rows = []
for feature, keywords in FEATURE_KEYWORDS.items():
    pattern = '|'.join(re.escape(k) for k in keywords)
    subset = reviews_df[reviews_df['review'].astype(str).str.contains(pattern, case=False, na=False)].head(3).copy()
    subset = subset[['drugName', 'condition', 'rating', 'review']]
    subset['review'] = subset['review'].astype(str).str.slice(0, 500)
    subset['feature_bucket'] = feature
    feature_rows.append(subset)

feature_preview = pd.concat(feature_rows, ignore_index=True)
feature_preview[['feature_bucket', 'drugName', 'condition', 'rating', 'review']]

## 4. Drug Name Coverage Against Hybrid Drug Table

这里回答一个关键问题：`hybrid` 推荐出来的药，能不能在评论表里查到？

In [ ]:
review_drugs = set(reviews_df['drug_key'])
table_drugs = set(drug_df['drug_key'])

coverage_summary = pd.DataFrame({
    'metric': [
        'unique_review_drugs',
        'unique_hybrid_table_drugs',
        'overlap_drugs',
        'hybrid_table_covered_by_review_ratio',
        'review_drugs_found_in_hybrid_table_ratio',
        'hybrid_table_only_count',
        'review_only_count',
    ],
    'value': [
        len(review_drugs),
        len(table_drugs),
        len(review_drugs & table_drugs),
        round(len(review_drugs & table_drugs) / len(table_drugs), 4),
        round(len(review_drugs & table_drugs) / len(review_drugs), 4),
        len(table_drugs - review_drugs),
        len(review_drugs - table_drugs),
    ]
})
coverage_summary

In [ ]:
hybrid_table_only = pd.DataFrame(sorted(table_drugs - review_drugs), columns=['drug_name_missing_in_review'])
review_only = pd.DataFrame(sorted(review_drugs - table_drugs), columns=['drug_name_missing_in_hybrid_table'])

print('Examples: in hybrid table but not in review table')
display(hybrid_table_only.head(30))
print('Examples: in review table but not in hybrid table')
display(review_only.head(30))

In [ ]:
covered_drug_counts = (
    reviews_df.groupby('drug_key')
    .size()
    .reset_index(name='review_count')
    .merge(drug_df[['drug_key', 'drug_name']], on='drug_key', how='inner')
    .sort_values('review_count', ascending=False)
)

covered_drug_counts.head(20)

## 5. Condition-Level Coverage

如果后面你想做 `drug + condition` 级别的评论特征表，这一块可以先看评论中的 condition 是否也能支持。

In [ ]:
drug_condition_counts = (
    reviews_df.groupby(['drug_key', 'condition_clean'])
    .size()
    .reset_index(name='review_count')
    .sort_values('review_count', ascending=False)
)

drug_condition_counts.head(20)

In [ ]:
condition_span = (
    reviews_df.groupby('drug_key')['condition_clean']
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name='condition_count')
)

condition_span.head(20)

## 6. Initial Takeaways

你可以重点回答这几个问题：
1. 评论文本是否足够丰富，能支持 feature engineering？
2. 评论药名是否足够覆盖 hybrid 结果？
3. 后续应该按 `drug` 聚合，还是按 `drug + condition` 聚合？
4. 优先做哪些 feature：`efficacy / side_effects / overall / cost / time_to_effect`？

通常如果你后面是用来 rerank `hybrid` 结果，这个 notebook 看完后就能决定是否值得做一张新的 comment feature table。